# ML-07 — Rule Baseline and First Ranked Queue

This notebook preserves our exact Week-4 rule-based baseline (`score` from `0` to `5`), reason codes (`STALE_HIGH_VOLUME`, `REVIEW`), and action tiers (`REVIEW_NOW`, `PRIORITIZE`, `MONITOR`) on `ml07_baseline_practice_dataset.csv` ($N = 30$) and benchmarks the same rule on `w05_ml_practice_dataset.csv` ($N = 100$).

## 1. The rule (and the 3-4 signals it uses)

Our Week-4 rule combines three pre-decision signals into an additive integer score (`0` to `5`):

```python
score = 2 * (impressions >= 1000) + 2 * (staleness_days >= 14) + 1 * (position >= 8)
```

- **Reason Code:** `STALE_HIGH_VOLUME` when `impressions >= 1000` and `staleness_days >= 14`, else `REVIEW`.
- **Action Tier:** `REVIEW_NOW` (`score >= 4`), `PRIORITIZE` (`score == 3`), `MONITOR` (`score < 3`).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
w04_path = REPO_ROOT / "work" / "data" / "ml07_baseline_practice_dataset.csv"
df = pd.read_csv(w04_path)

df["score"] = (
    (df["impressions"] >= 1000).astype(int) * 2
    + (df["staleness_days"] >= 14).astype(int) * 2
    + (df["position"] >= 8).astype(int) * 1
)
df["reason_code"] = "REVIEW"
df.loc[(df["impressions"] >= 1000) & (df["staleness_days"] >= 14), "reason_code"] = "STALE_HIGH_VOLUME"

df["action"] = "MONITOR"
df.loc[df["score"] >= 3, "action"] = "PRIORITIZE"
df.loc[df["score"] >= 4, "action"] = "REVIEW_NOW"

print("Scored Week-4 Practice Dataset (first 10 rows):")
print(df.head(10).to_string(index=False))


Scored Week-4 Practice Dataset (first 10 rows):
 item_id  impressions  clicks  position  staleness_days    ctr  score       reason_code     action
item_001          537     227        13              14 0.4227      3            REVIEW PRIORITIZE
item_002         3892     117        11              25 0.0301      5 STALE_HIGH_VOLUME REVIEW_NOW
item_003         3307      50         6              26 0.0151      4 STALE_HIGH_VOLUME REVIEW_NOW
item_004         2250     279        14              12 0.1240      3            REVIEW PRIORITIZE
item_005         2221     444         6              27 0.1999      4 STALE_HIGH_VOLUME REVIEW_NOW
item_006         4307      36         5               9 0.0084      2            REVIEW    MONITOR
item_007          521     429        13               8 0.8234      1            REVIEW    MONITOR
item_008         3517     414         6              21 0.1177      4 STALE_HIGH_VOLUME REVIEW_NOW
item_009         1087     141         2              20 0.129

## 2. Top-10 ranked queue (with reason codes)

*Sorted by `score` descending (stable `mergesort` preserving original item order among ties), exactly matching our Week-4 submission.*

In [2]:
df_ranked = df.sort_values("score", ascending=False, kind="mergesort").reset_index(drop=True)
df_ranked["rank"] = range(1, len(df_ranked) + 1)

top10 = df_ranked.head(10)[["rank", "item_id", "impressions", "clicks", "position", "staleness_days", "ctr", "score", "reason_code", "action"]]
print("Preserved Week-4 Top-10 Refresh Queue (N=30):")
print(top10.to_string(index=False))


Preserved Week-4 Top-10 Refresh Queue (N=30):
 rank  item_id  impressions  clicks  position  staleness_days    ctr  score       reason_code     action
    1 item_002         3892     117        11              25 0.0301      5 STALE_HIGH_VOLUME REVIEW_NOW
    2 item_011         2679      86        12              25 0.0321      5 STALE_HIGH_VOLUME REVIEW_NOW
    3 item_015         3615      38        10              24 0.0105      5 STALE_HIGH_VOLUME REVIEW_NOW
    4 item_019         4214     340         8              15 0.0807      5 STALE_HIGH_VOLUME REVIEW_NOW
    5 item_020         2306     390        10              22 0.1691      5 STALE_HIGH_VOLUME REVIEW_NOW
    6 item_024         4641     236        12              14 0.0509      5 STALE_HIGH_VOLUME REVIEW_NOW
    7 item_025         3929     251         9              16 0.0639      5 STALE_HIGH_VOLUME REVIEW_NOW
    8 item_026         3254      26        10              18 0.0080      5 STALE_HIGH_VOLUME REVIEW_NOW
    9 ite

## 3. How good is the rule? (Precision@10 and Test Split Benchmark)

*Because `ml07_baseline_practice_dataset.csv` ($N=30$) is an unlabeled triage practice table, we also evaluate the exact same Week-4 rule (`score >= 3`) on the labeled `w05_ml_practice_dataset.csv` ($N=100$, test split $n=20$) to measure its classification accuracy, F1, Precision@10, and ROC AUC.*

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df_w05 = pd.read_csv(REPO_ROOT / "work" / "data" / "w05_ml_practice_dataset.csv")
_, Xte, _, yte = train_test_split(
    df_w05[["impressions", "clicks", "staleness_days", "position"]],
    df_w05["target"].astype(int),
    test_size=0.20,
    random_state=42,
    stratify=df_w05["target"].astype(int),
)

b_score = (
    (Xte["impressions"] >= 1000).astype(int) * 2
    + (Xte["staleness_days"] >= 14).astype(int) * 2
    + (Xte["position"] >= 8).astype(int) * 1
)
b_pred = (b_score >= 3).astype(int)

print("Week-4 Rule Baseline Performance on Labeled W05 Test Split (n=20, Base Rate = 0.6000):")
print(f"  Accuracy : {accuracy_score(yte, b_pred):.4f}")
print(f"  Precision: {precision_score(yte, b_pred):.4f}")
print(f"  Recall   : {recall_score(yte, b_pred):.4f}")
print(f"  F1 Score : {f1_score(yte, b_pred):.4f}")
print(f"  ROC AUC  : {roc_auc_score(yte, b_score):.4f}")


Week-4 Rule Baseline Performance on Labeled W05 Test Split (n=20, Base Rate = 0.6000):
  Accuracy : 0.7000
  Precision: 0.6875
  Recall   : 0.9167
  F1 Score : 0.7857
  ROC AUC  : 0.8542


## 4. Where the rule breaks (and why we need a learned model next)

1. **Coarse Integer Ties:** 8 of the Top 10 items in `ml07_baseline_practice_dataset.csv` tie at `score = 5`, providing no granular ranking within the top bucket.
2. **Ignores Clicks / CTR:** Items like `item_005` (`ctr = 19.99%`, `position = 6`) and `item_020` (`ctr = 16.91%`) receive `REVIEW_NOW` solely from impressions and staleness even though they already capture high clicks.

In [4]:
print("Action Tier Distribution on W04 Practice Dataset (N=30):")
print(df_ranked["action"].value_counts().to_string())


Action Tier Distribution on W04 Practice Dataset (N=30):
action
REVIEW_NOW    16
MONITOR        9
PRIORITIZE     5


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`